In [1]:
import pydantic
print(pydantic.__version__)

2.12.5


# Validation without Pydantic

In [2]:
class User:
    def __init__(self, id : int, name="John Doe"):
        if not isinstance(id, int):
            raise TypeError(f'Expected id to be an int, got {type(id).__name__}')
        
        if not isinstance(name, str):
            raise TypeError(f'Expected name to be str, got{type(name).__name__}')
        
        self.id = id
        self.name = name

try:
    user = User(id='123')
except TypeError as e :
    print(e)

Expected id to be an int, got str


# Validation with Pydantic

In [3]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str = 'John Doe'

In [4]:
user = User(id='123')

In [5]:
print(user.id)

123


In [6]:
print(user.model_fields_set)
user = User(id = '123', name="Joe Doe")
print(user.model_fields_set)

{'id'}
{'name', 'id'}


In [7]:
print(user.model_dump())
print(user.model_dump_json())
print(user.model_json_schema())

{'id': 123, 'name': 'Joe Doe'}
{"id":123,"name":"Joe Doe"}
{'properties': {'id': {'title': 'Id', 'type': 'integer'}, 'name': {'default': 'John Doe', 'title': 'Name', 'type': 'string'}}, 'required': ['id'], 'title': 'User', 'type': 'object'}


## Nested Models

In [8]:
from typing import List, Optional
from pydantic import BaseModel

class Food(BaseModel):
    name: str
    price: float
    ingredients: Optional[List[str]] = None

class Restaurant(BaseModel):
    name: str
    location: str
    foods: List[Food]

restaurant_instance = Restaurant(
    name="Tasty Bites",
    location="4th Floor, President Plaza",
    foods=[
        {"name":"Cheese Pizza", "price":12.50, "ingredients": ["Cheese", "Tomato Sauce", "Dough"]},
        {"name": "Veggie Burger", "price": 8.99}
    ]
)

print(restaurant_instance)
print(restaurant_instance.model_dump())

name='Tasty Bites' location='4th Floor, President Plaza' foods=[Food(name='Cheese Pizza', price=12.5, ingredients=['Cheese', 'Tomato Sauce', 'Dough']), Food(name='Veggie Burger', price=8.99, ingredients=None)]
{'name': 'Tasty Bites', 'location': '4th Floor, President Plaza', 'foods': [{'name': 'Cheese Pizza', 'price': 12.5, 'ingredients': ['Cheese', 'Tomato Sauce', 'Dough']}, {'name': 'Veggie Burger', 'price': 8.99, 'ingredients': None}]}


In [9]:
!pip install pydantic[email]

  Using cached email_validator-2.3.0-py3-none-any.whl.metadata (26 kB)
  Using cached dnspython-2.8.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
Using cached email_validator-2.3.0-py3-none-any.whl (35 kB)
Using cached dnspython-2.8.0-py3-none-any.whl (331 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)

   ---------------------------------------- 0/3 [idna]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 

In [10]:
from typing import List
from pydantic import BaseModel, EmailStr, PositiveInt, conlist, Field, HttpUrl

class Address(BaseModel):
    street: str
    city: str
    state: str
    zip_code: str

class Employee(BaseModel):
    name: str
    position: str
    email: EmailStr

class Owner(BaseModel):
    name: str
    email: EmailStr

class Restaurant(BaseModel):
    name: str = Field(..., pattern=r"^[a-zA-Z0-9-' ]+$")
    owner : Owner
    address: Address
    employees: conlist(Employee, min_length=2)
    number_of_seats: PositiveInt
    delivery: bool
    website: HttpUrl

# Creating an instance of the Restaurant class
restaurant_instance = Restaurant(
    name="Tasty Btes",
    owner={
        "name": "John Doe",
        "email": "john.doe@example.com"
    },
    address={
        "street": "4th Floor, President Plaza",
        "city": "Ahmedabad",
        "state": "Gujarat",
        "zip_code": "12345"
    },
    employees=[
        {
            "name": "Joe Doe",
            "position": "Chef",
            "email": "joe.doe@example.com"
        },
        {
            "name": "Mike Roe",
            "position": "Waiter",
            "email": "mile.roe@example.com"
        }
    ],
    number_of_seats=50,
    delivery=True,
    website="http://tastybites.com"
    
)

# Printing the instance
print(restaurant_instance)


name='Tasty Btes' owner=Owner(name='John Doe', email='john.doe@example.com') address=Address(street='4th Floor, President Plaza', city='Ahmedabad', state='Gujarat', zip_code='12345') employees=[Employee(name='Joe Doe', position='Chef', email='joe.doe@example.com'), Employee(name='Mike Roe', position='Waiter', email='mile.roe@example.com')] number_of_seats=50 delivery=True website=HttpUrl('http://tastybites.com/')


## Field Validators

In [11]:
from pydantic import BaseModel, EmailStr, field_validator

class Owner(BaseModel):
    name: str
    email: EmailStr

    @field_validator('name')
    @classmethod
    def name_must_contain_space(cls, v: str) -> str:
        if ' ' not in v:
            raise ValueError ('Owner name must contain a space')
        return v.title()
    
try:
    owner_instance = Owner(name="john doe", email="john.doe@example.com")
except ValueError as e:
    print(e)

print(owner_instance)

name='John Doe' email='john.doe@example.com'


## Model validators - allowing you to create a model before and after field validation 

In [12]:
from typing import Any
from pydantic import BaseModel, EmailStr, ValidationError, model_validator

class Owner(BaseModel):
    name: str
    email: EmailStr

    @model_validator(mode='before')
    @classmethod
    def check_sensitive_info_omitted(cls, data: Any) -> Any:
        if isinstance(data, dict):
            if 'password' in data:
                raise ValueError ('password should not be included')
            if 'card_number' in data:
                raise ValueError('carsd_number should not be included')
        return data
    
    @model_validator(mode='after')
    def check_name_contains_space(self) -> 'Owner':
        if ' ' not in self.name:
            raise ValueError('Owner name must contain a space')
        return self
    
try :
    Owner(name="john doe", email="john.doe@example.com")
except ValueError as e:
    print(e)

## Fields - Used to customize and add metadata to fields of models

In [13]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(default='John Doe')

user = User()
print(user)

name='John Doe'


In [14]:
from uuid import uuid4

from pydantic import BaseModel, Field

class User(BaseModel):
    id: int = Field(default_factory= lambda: uuid4().hex)

user = User()
print(user)

id='2037c7884fd54c32b4dbd1a9df4aeaab'


## Field aliases - For validation and seralization, you can define an alias for a field.

### There are three ways to define an alias:

* `Field(..., alias='Foo')`
* `Field(..., validation_alias='Foo')`
* `Field(..., serialization_alias='Foo')`

In [15]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(..., alias='username')

user = User(username='johndoe')
print(user)
print(user.model_dump(by_alias=True))

name='johndoe'
{'username': 'johndoe'}


#### But why would you want to do it? for example if your API and database fields differ - you only need one model 

### Let's look at field contraints 

In [16]:
from typing import List
from pydantic import BaseModel, Field, EmailStr
from decimal import Decimal

class User(BaseModel):
    username: str = Field(..., min_length=3, max_length=10, pattern=r'^\w+$')
    email: EmailStr = Field(...)
    age: int = Field(..., gt=0, le=120)
    height: float = Field(..., gt = 0.0)
    is_active: bool = Field(True)
    balance: Decimal = Field(..., max_digits=10, decimal_places=2)
    favorie_numbers: List[int] = Field(..., min_items=1)

C:\Users\praja\AppData\Local\Temp\ipykernel_26712\3152090584.py:12: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  favorie_numbers: List[int] = Field(..., min_items=1)


In [17]:
user_instance = User(
    username= "johndoe",
    age= 30,
    height= 5.11,
    weight= 60.5,
    email= "john.doe@example.com",
    password = "securepassword",
    balance= 9999.99, 
    favorie_numbers=[7, 69]
)

print(user_instance.model_dump)

<bound method BaseModel.model_dump of User(username='johndoe', email='john.doe@example.com', age=30, height=5.11, is_active=True, balance=Decimal('9999.99'), favorie_numbers=[7, 69])>


### Computed Fields 

In [18]:
from pydantic import BaseModel, computed_field
from datetime import datetime

class Person(BaseModel):
    name: str
    birth_year: int

    @computed_field
    @property
    def age(self) -> int:
        current_year = datetime.now().year
        return current_year - self.birth_year
    
print(Person(name="John Doe", birth_year=2000).model_dump())

{'name': 'John Doe', 'birth_year': 2000, 'age': 26}


In [19]:
from pydantic import BaseModel, ValidationError, field_validator
from datetime import datetime

class Person(BaseModel):
    name: str
    birth_year: int

    @property
    def age(self) -> int:
        current_year = datetime.now().year
        return current_year - self.birth_year
    
    @field_validator('birth_year')
    @classmethod
    def validate_age(cls, v: int) -> int:
        current_year = datetime.now().year
        if current_year - v < 18 :
            raise ValueError('Person must be 18 years or older')
        return v
    
try:
    print(Person(name= "John Doe", birth_year=2000))
except ValueError as e:
    print(e)

name='John Doe' birth_year=2000


#### You also ue dataclasses and pydantics validation logic - dataclasses do not provide that out of the box

In [20]:
from dataclasses import dataclass,  field
from typing import List, Optional
from pydantic import Field, TypeAdapter

@dataclass
class User:
    id: int
    name: str = 'John Doe'
    age: Optional[int] = field(
        default= None,
        metadata= dict(title= 'The age of the user', description= "do not lie!", ge= 18),
    )
    height: Optional[int] = Field(None, title='The height in cm', ge=50, le=300)
    friends: List[int] = field(default_factory=lambda: [0])

#### TypeAdapter

#### You may have types that are not `BaseModels` that you want to validate data against. Or you may want to validate a `List[SomeModel]` or dump it to JSON.

#### For use cases like this, Pydantic provides `TypeAdapter`, which can be used for type validation, serialization, and JSON schema generation  without creating a `BaseModel`.

In [21]:
#Example of using TypeAdapter to get json_schema of the User dataclass
print(TypeAdapter(User).json_schema())

{'properties': {'id': {'title': 'Id', 'type': 'integer'}, 'name': {'default': 'John Doe', 'title': 'Name', 'type': 'string'}, 'age': {'anyOf': [{'minimum': 18, 'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'do not lie!', 'title': 'The age of the user'}, 'height': {'anyOf': [{'maximum': 300, 'minimum': 50, 'type': 'integer'}, {'type': 'null'}], 'default': None, 'title': 'The height in cm'}, 'friends': {'items': {'type': 'integer'}, 'title': 'Friends', 'type': 'array'}}, 'required': ['id'], 'title': 'User', 'type': 'object'}


## Strict Mode

#### Pydantic, by default, tries to coerce values to the declared type, converting inputs like the string "123" to integer 123, but this automatic conversion can be undesirable in situatons where strict type compliance is required, causing a need for cofigurations to make Pydantic error out instead.

In [22]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    id: int
    username: str

print(User.model_validate({'id': '42', 'username': 'john_doe'})) # lax mode
#> id= 42 username= 'john_doe'


id=42 username='john_doe'


In [23]:
try:
    User.model_validate({'id':'42', 'username': 'john_doe'}, strict=True)
except ValidationError as exc:
    print(exc)

1 validation error for User
id
  Input should be a valid integer [type=int_type, input_value='42', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_type


## Settings Management 

#### Pydantic Settings provides optional Pydantic features for loading a settings or config class from enviroment variables or secrets files.

In [24]:
!pip install pydantic-settings

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------------------------------------- 0/2 [python-dotenv]
   -------------------- ------------------- 1/2 [pydantic-settings]
   -------------------- ------------------- 1/2 [pydantic-settings]
   -------------------- ------------------- 1/2 [pydantic-settings]
   ---------------------------------------- 2/2 [pydantic-settings]



In [25]:
# import os 
# from pydantic import Field
# from pydantic_settings import BaseSettings

# class Settings(BaseSettings):
#     auth_key: str = Field(...)
#     api_key: str = Field(alias='my_api_key')

# print(Settings().model_dump())


from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    # 1. 'env_file' tells Pydantic to read your .env file
    # 2. 'extra="ignore"' tells it to ignore variables not defined here (like env2)
    model_config = SettingsConfigDict(env_file='.env', extra='ignore')

    auth_key: str = Field(...)
    
    # Use the alias 'my_api_key' in your .env file
    api_key: str = Field(alias='my_api_key')

# This will now load values from your .env file automatically
try:
    settings = Settings()
    print(settings.model_dump())
except Exception as e:
    print(f"Validation Error: {e}")



Validation Error: 2 validation errors for Settings
auth_key
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
my_api_key
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


In [26]:
# import os
# from pydantic import Field, AliasChoices
# from pydantic_settings import BaseSettings

# os.environ["AUTH_KEY"] = "test_auth_key"
# os.environ["MY_API_KEY"] = "test"
# os.environ["ENV2"] = "https://mysuperurl.com"

# class Settings(BaseSettings):
#     service_name: str = Field(default= "default")
#     auth_key: str
#     api_key: str = Field(alias='my_api_key')
#     url: str = Field(validate_alias = AliasChoices("env1", "env2"))

# print(Settings().model_dump())


import os 
from pydantic import Field, AliasChoices
from pydantic_settings import BaseSettings, SettingsConfigDict

# These are uppercase
os.environ["AUTH_KEY"] = "test_auth_key"
os.environ["MY_API_KEY"] = "test"
os.environ["ENV2"] = "https://mysuperurl.com"

class Settings(BaseSettings):
    # ADD THIS: This tells Pydantic to match uppercase ENV vars to lowercase aliases
    model_config = SettingsConfigDict(case_sensitive=False)

    service_name: str = Field(default="default")
    auth_key: str
    api_key: str = Field(alias='my_api_key')
    url: str = Field(validation_alias=AliasChoices("env1", "env2")) # Note: 'validation_alias' in V2

print(Settings().model_dump())


{'service_name': 'default', 'auth_key': 'test_auth_key', 'api_key': 'test', 'url': 'https://mysuperurl.com'}


In [27]:
import os 
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

#Set enviroment variables with the prefix
os.environ["PRODUCTION_AUTH_KEY"] = "test_auth_key"
os.environ["PRODUCTION_MY_API_KEY"] = "test"
os.environ["PRODUCTION_ENV2"] = "https://mysuperurl.com"

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix='production_')

    service_name: str = Field(default="default")
    auth_key: str
    api_key: str = Field(alias='my_api_key')
    url: str = Field(validation_alias= AliasChoices("env1", "env2"))

print(Settings().model_dump())

{'service_name': 'default', 'auth_key': 'test_auth_key', 'api_key': 'test', 'url': 'https://mysuperurl.com'}


### You can also use .env file 

In [28]:
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):

    model_config = SettingsConfigDict(env_file='.env', env_file_encoding='utf-8', extra='ignore')

    service_name: str = Field(default="default")
    auth_key: str
    api_key: str = Field(alias='my_api_key')

print(Settings().model_dump())

{'service_name': 'default', 'auth_key': 'test_auth_key', 'api_key': 'test'}
